# 🔗 Production-Ready RAG Pipeline with Deduplication & Versioning

This notebook builds a complete **Retrieval-Augmented Generation (RAG)** pipeline from local files.
It handles real-world data engineering problems:
- ✅ Duplicate file detection via SHA256 hashing
- ✅ Chunk-level deduplication
- ✅ File versioning & re-ingestion safety
- ✅ Metadata tracking per chunk
- ✅ ChromaDB via Docker REST API
- ✅ Sentence-Transformers embeddings (`all-MiniLM-L6-v2`)
- ✅ RAG retrieval + prompt builder

**Prerequisites:** ChromaDB running on `localhost:8000` via Docker.

---
## Cell 1 — Install Required Libraries

In [ ]:
# Install all required dependencies
# Run this cell once; restart kernel if needed after installation

%pip install --quiet \
    chromadb \
    sentence-transformers \
    langchain \
    langchain-community \
    langchain-text-splitters \
    pypdf \
    python-docx \
    requests \
    tqdm \
    python-dotenv \
    numpy

print("✅ All libraries installed successfully.")

---
## Cell 2 — Imports & Global Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import json
import hashlib
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
from tqdm import tqdm

# ── Document loaders ──────────────────────────────────────────────────────────
import pypdf                          # PDF parsing
from docx import Document as DocxDoc  # DOCX parsing

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Embeddings ────────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

# ── ChromaDB (HTTP client — connects to Docker) ───────────────────────────────
import chromadb
from chromadb.config import Settings

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL CONFIGURATION  (edit these to match your environment)
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    # Folder that contains your PDF / TXT / DOCX files
    "data_dir": "../data-RAG",

    # ChromaDB Docker endpoint
    "chroma_host": "localhost",
    "chroma_port": 8000,

    # Collection name inside ChromaDB
    "collection_name": "rag_collection",

    # Chunking parameters
    "chunk_size": 800,
    "chunk_overlap": 100,

    # Sentence-Transformers model
    # Why all-MiniLM-L6-v2?
    #   • Very fast inference (~5ms / sentence on CPU)
    #   • 384-dim embeddings — lightweight for production
    #   • Strong semantic understanding for retrieval tasks
    #   • Open-source, no API key required
    "embedding_model": "all-MiniLM-L6-v2",

    # Local JSON registry that tracks ingested files (deduplication log)
    "registry_path": "./ingestion_registry.json",

    # Similarity threshold for near-duplicate detection (cosine)
    "near_dup_threshold": 0.97,

    # Retrieval: number of results to return
    "top_k": 5,
}

print("✅ Configuration loaded:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

---
## Cell 3 — Registry: Load / Save Ingestion State

The **ingestion registry** is a local JSON file that maps each file hash to its
metadata. This is the single source of truth for deduplication and versioning.
It survives process restarts and ChromaDB wipes.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ingestion Registry Helpers
# Structure: { file_hash: { filename, path, version, ingested_at, chunk_ids } }
# ─────────────────────────────────────────────────────────────────────────────

def load_registry(path: str) -> dict:
    """Load the JSON registry from disk; return empty dict if not found."""
    registry_path = Path(path)
    if registry_path.exists():
        with open(registry_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_registry(registry: dict, path: str) -> None:
    """Persist the registry back to disk (atomic-ish write)."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(registry, f, indent=2, ensure_ascii=False)


def get_file_version(registry: dict, filename: str) -> int:
    """
    Return the next version number for a given filename.
    Scans all registry entries for this filename (different hashes = new version).
    """
    versions = [
        entry["version"]
        for entry in registry.values()
        if entry["filename"] == filename
    ]
    return max(versions) + 1 if versions else 1


# Load registry at startup
REGISTRY = load_registry(CONFIG["registry_path"])

print(f"✅ Registry loaded. Tracked files: {len(REGISTRY)}")

---
## Cell 4 — Hashing Utilities

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Hashing Utilities
#   • sha256_file   → hash the raw bytes of a file (deduplication at file level)
#   • sha256_string → hash a text chunk (deduplication at chunk level)
# ─────────────────────────────────────────────────────────────────────────────

def sha256_file(filepath: str) -> str:
    """
    Compute SHA-256 hash of a file from raw bytes.
    Reading in 64 KB blocks keeps memory usage constant for large files.
    """
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            h.update(block)
    return h.hexdigest()


def sha256_string(text: str) -> str:
    """Compute SHA-256 hash of a UTF-8 string (used for chunk deduplication)."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def is_file_already_ingested(file_hash: str, registry: dict) -> bool:
    """Return True if this exact file hash is already in the registry."""
    return file_hash in registry


# Quick smoke-test
test_hash = sha256_string("hello world")
print(f"✅ SHA-256 of 'hello world': {test_hash[:32]}...")

---
## Cell 5 — Document Loaders (PDF / TXT / DOCX)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Document Loaders
#   Each loader returns a list of dicts: { "text": str, "page": int|None }
#   Page numbers let us reconstruct provenance in metadata.
# ─────────────────────────────────────────────────────────────────────────────

def load_pdf(filepath: str) -> list[dict]:
    """Extract text page-by-page from a PDF using pypdf."""
    pages = []
    try:
        reader = pypdf.PdfReader(filepath)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                pages.append({"text": text, "page": page_num})
    except Exception as e:
        print(f"  ⚠️  Could not read PDF '{filepath}': {e}")
    return pages


def load_txt(filepath: str) -> list[dict]:
    """Read a plain-text file as a single document block (no page concept)."""
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            text = f.read()
        if text.strip():
            return [{"text": text, "page": None}]
    except Exception as e:
        print(f"  ⚠️  Could not read TXT '{filepath}': {e}")
    return []


def load_docx(filepath: str) -> list[dict]:
    """Extract paragraph text from a DOCX file."""
    try:
        doc = DocxDoc(filepath)
        full_text = "\n".join(
            para.text for para in doc.paragraphs if para.text.strip()
        )
        if full_text.strip():
            return [{"text": full_text, "page": None}]
    except Exception as e:
        print(f"  ⚠️  Could not read DOCX '{filepath}': {e}")
    return []


def load_document(filepath: str) -> list[dict]:
    """Route a file to the appropriate loader based on its extension."""
    ext = Path(filepath).suffix.lower()
    if ext == ".pdf":
        return load_pdf(filepath)
    elif ext == ".txt":
        return load_txt(filepath)
    elif ext == ".docx":
        return load_docx(filepath)
    else:
        print(f"  ⚠️  Unsupported file type: {ext} — skipping '{filepath}'")
        return []


def discover_files(data_dir: str) -> list[str]:
    """Recursively find all PDF, TXT, DOCX files under data_dir."""
    supported_ext = {".pdf", ".txt", ".docx"}
    files = [
        str(p)
        for p in Path(data_dir).rglob("*")
        if p.is_file() and p.suffix.lower() in supported_ext
    ]
    return sorted(files)


# Discover files in the data folder
all_files = discover_files(CONFIG["data_dir"])
print(f"✅ Discovered {len(all_files)} supported file(s):")
for f in all_files:
    print(f"   {f}")

---
## Cell 6 — Text Cleaning

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Text Cleaning Pipeline
#   Applied BEFORE chunking to improve embedding quality.
#   Order of operations matters: unicode → whitespace → special chars
# ─────────────────────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """
    Normalise raw extracted text for embedding.

    Steps:
      1. Unicode normalisation (NFKC) — collapses ligatures, normalises chars
      2. Replace non-breaking spaces and zero-width chars
      3. Remove control characters (except newline / tab)
      4. Collapse multiple blank lines to a single blank line
      5. Strip leading/trailing whitespace from each line
      6. Collapse multiple spaces to one
      7. Final strip
    """
    # 1. Unicode normalisation
    text = unicodedata.normalize("NFKC", text)

    # 2. Replace common unicode noise
    text = text.replace("\u00a0", " ")   # non-breaking space
    text = text.replace("\u200b", "")    # zero-width space
    text = text.replace("\ufeff", "")    # BOM

    # 3. Strip control characters (keep \n \t)
    text = "".join(
        ch if (unicodedata.category(ch) != "Cc" or ch in "\n\t")
        else " "
        for ch in text
    )

    # 4. Collapse 3+ consecutive newlines → 2 (paragraph boundary)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 5. Strip each line
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(lines)

    # 6. Collapse intra-line multiple spaces
    text = re.sub(r"[ \t]{2,}", " ", text)

    # 7. Final strip
    return text.strip()


def remove_repeated_header_footer(pages: list[dict], min_repeat: int = 3) -> list[dict]:
    """
    Heuristic: if the same line appears verbatim in at least `min_repeat` pages
    it is likely a header or footer. Remove those lines from every page.

    Works best for PDFs with consistent headers/footers.
    """
    if len(pages) < min_repeat:
        return pages

    # Count frequency of each non-empty line across pages
    from collections import Counter
    line_counts: Counter = Counter()
    for page in pages:
        for line in page["text"].splitlines():
            stripped = line.strip()
            if stripped:
                line_counts[stripped] += 1

    # Lines appearing in >= min_repeat pages are considered noise
    noise_lines = {line for line, count in line_counts.items() if count >= min_repeat}

    cleaned = []
    for page in pages:
        filtered_lines = [
            line for line in page["text"].splitlines()
            if line.strip() not in noise_lines
        ]
        cleaned.append({"text": "\n".join(filtered_lines), "page": page["page"]})
    return cleaned


# Smoke-test
sample_dirty = "  Hello\u00a0World!\n\n\n   \t Extra spaces  \n"
print("Before:", repr(sample_dirty))
print("After :", repr(clean_text(sample_dirty)))
print("\n✅ Text cleaner ready.")

---
## Cell 7 — Chunking with Deduplication

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chunking
#   • RecursiveCharacterTextSplitter respects paragraph / sentence boundaries
#   • Each chunk gets a SHA-256 hash → used to prevent re-insertion
# ─────────────────────────────────────────────────────────────────────────────

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def build_chunks(
    pages: list[dict],
    filepath: str,
    file_hash: str,
    file_type: str,
    version: int,
    existing_chunk_hashes: set[str],
) -> list[dict]:
    """
    Split page texts into chunks and attach rich metadata.

    Returns a list of chunk dicts ready for ChromaDB insertion.
    Chunks whose content hash already exists are silently skipped.
    """
    filename = Path(filepath).name
    ingestion_ts = datetime.now(timezone.utc).isoformat()
    chunks_out = []
    chunk_index = 0
    seen_hashes_this_run: set[str] = set()  # deduplicate within this file

    for page_data in pages:
        raw_text = page_data["text"]
        page_num = page_data["page"]  # int or None

        split_texts = splitter.split_text(raw_text)

        for text in split_texts:
            text = text.strip()
            if not text:
                continue

            chunk_hash = sha256_string(text)

            # Skip if this chunk already exists in ChromaDB OR seen in this run
            if chunk_hash in existing_chunk_hashes or chunk_hash in seen_hashes_this_run:
                continue

            seen_hashes_this_run.add(chunk_hash)

            # Unique Chroma ID: file_hash prefix + chunk index
            chunk_id = f"{file_hash[:12]}_{chunk_index:04d}"

            metadata = {
                "filename":         filename,
                "filepath":         str(filepath),
                "file_type":        file_type,
                "file_hash":        file_hash,
                "chunk_id":         chunk_id,
                "chunk_hash":       chunk_hash,
                "page":             page_num if page_num is not None else -1,
                "ingestion_ts":     ingestion_ts,
                "version":          version,
            }

            chunks_out.append({
                "id":       chunk_id,
                "text":     text,
                "metadata": metadata,
            })
            chunk_index += 1

    return chunks_out


print(f"✅ RecursiveCharacterTextSplitter configured "
      f"(chunk_size={CONFIG['chunk_size']}, overlap={CONFIG['chunk_overlap']})")

---
## Cell 8 — Embedding Model

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Embedding Model: all-MiniLM-L6-v2
#
# Why this model?
#   • 384-dimensional dense vectors — small enough for fast ANN search
#   • Trained specifically for semantic similarity / retrieval tasks
#   • ~22 MB on disk — trivially deployable without a GPU
#   • Excellent trade-off between quality and inference speed on CPU
# ─────────────────────────────────────────────────────────────────────────────

print(f"🔄 Loading embedding model '{CONFIG['embedding_model']}' ...")
embedding_model = SentenceTransformer(CONFIG["embedding_model"])
print(f"✅ Model loaded. Output dimension: {embedding_model.get_sentence_embedding_dimension()}")


def embed_texts(texts: list[str], batch_size: int = 64) -> list[list[float]]:
    """
    Encode a list of strings into embedding vectors.
    Returns a list of plain Python float lists (ChromaDB-compatible).
    """
    vectors = embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,   # L2-normalise → dot product = cosine similarity
    )
    return vectors.tolist()


# Smoke-test
test_vec = embed_texts(["This is a test sentence."])
print(f"   Sample embedding — first 5 dims: {[round(v, 4) for v in test_vec[0][:5]]}")

---
## Cell 9 — ChromaDB Connection & Collection Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ChromaDB — HTTP Client (connects to the Docker container)
#
# We use get_or_create_collection so the pipeline is idempotent:
# running it multiple times won't create duplicate collections.
# ─────────────────────────────────────────────────────────────────────────────

print(f"🔄 Connecting to ChromaDB at "
      f"{CONFIG['chroma_host']}:{CONFIG['chroma_port']} ...")

chroma_client = chromadb.HttpClient(
    host=CONFIG["chroma_host"],
    port=CONFIG["chroma_port"],
)

# Verify connection
try:
    chroma_client.heartbeat()
    print("✅ ChromaDB heartbeat OK.")
except Exception as e:
    raise ConnectionError(
        f"Cannot reach ChromaDB at {CONFIG['chroma_host']}:{CONFIG['chroma_port']}. "
        f"Is the Docker container running?\nOriginal error: {e}"
    )

# Create or reuse the collection
collection = chroma_client.get_or_create_collection(
    name=CONFIG["collection_name"],
    metadata={"hnsw:space": "cosine"},   # cosine similarity for retrieval
)

print(f"✅ Collection '{CONFIG['collection_name']}' ready. "
      f"Current doc count: {collection.count()}")


def get_existing_chunk_hashes() -> set[str]:
    """
    Fetch all chunk_hash metadata values from the collection.
    Used at pipeline startup to avoid re-inserting already-stored chunks.

    Note: for very large collections (>500 k chunks) consider a dedicated
    hash index (Redis SET / SQLite) instead of fetching all metadata.
    """
    total = collection.count()
    if total == 0:
        return set()

    # ChromaDB paginates; fetch in batches of 5000
    hashes: set[str] = set()
    offset = 0
    batch = 5000
    while offset < total:
        results = collection.get(
            limit=batch,
            offset=offset,
            include=["metadatas"],
        )
        for meta in results["metadatas"]:
            if "chunk_hash" in meta:
                hashes.add(meta["chunk_hash"])
        offset += batch

    return hashes


# Load all existing chunk hashes once (used by the ingestion loop below)
EXISTING_CHUNK_HASHES = get_existing_chunk_hashes()
print(f"   Existing chunk hashes loaded: {len(EXISTING_CHUNK_HASHES)}")

---
## Cell 10 — Near-Duplicate Detection (Optional, Embedding-Based)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Near-Duplicate Detection
#
# Problem: two files may have slightly different hashes (e.g. different PDF
# export settings) but near-identical semantic content. Exact hashing won't
# catch these. We embed the first 512 chars of the document and query Chroma
# for a near-match above a cosine threshold.
#
# If a near-duplicate is found we log a warning and still allow ingestion
# (the user may have intentionally updated the file). You can flip this to
# a hard skip by returning True instead of False in the positive branch.
# ─────────────────────────────────────────────────────────────────────────────

def is_near_duplicate(
    first_page_text: str,
    threshold: float = CONFIG["near_dup_threshold"],
) -> tuple[bool, Optional[str]]:
    """
    Query the collection with the document's first ~512 characters.
    Returns (is_near_dup, matching_filename_or_None).

    A cosine distance of 0 means identical; 1 means orthogonal.
    Chroma returns `distance` so we convert: similarity = 1 - distance.
    """
    if collection.count() == 0:
        return False, None

    probe_text = first_page_text[:512].strip()
    if not probe_text:
        return False, None

    vec = embed_texts([probe_text])
    results = collection.query(
        query_embeddings=vec,
        n_results=1,
        include=["metadatas", "distances"],
    )

    if not results["distances"] or not results["distances"][0]:
        return False, None

    distance   = results["distances"][0][0]
    similarity = 1.0 - distance
    meta       = results["metadatas"][0][0]

    if similarity >= threshold:
        return True, meta.get("filename", "unknown")

    return False, None


print(f"✅ Near-duplicate detector ready (cosine threshold = {CONFIG['near_dup_threshold']}).")

---
## Cell 11 — Main Ingestion Pipeline

This cell ties together all previous cells into a single, robust ingestion loop.
Running it multiple times is **safe** — files already in the registry are skipped.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ingestion Pipeline
# ─────────────────────────────────────────────────────────────────────────────

def ingest_file(filepath: str) -> dict:
    """
    Full ingestion pipeline for a single file.

    Returns a status dict::
        { status: 'ingested' | 'skipped_exact_dup' | 'warned_near_dup' | 'empty' | 'error',
          chunks_added: int }
    """
    filepath = str(filepath)
    filename = Path(filepath).name
    file_type = Path(filepath).suffix.lower().lstrip(".")

    # ── Step A: Compute file hash ──────────────────────────────────────────
    try:
        file_hash = sha256_file(filepath)
    except Exception as e:
        print(f"  ❌ Cannot hash '{filename}': {e}")
        return {"status": "error", "chunks_added": 0}

    # ── Step B: Exact duplicate check ─────────────────────────────────────
    if is_file_already_ingested(file_hash, REGISTRY):
        print(f"  ⏭️  SKIP (exact duplicate): {filename}")
        return {"status": "skipped_exact_dup", "chunks_added": 0}

    # ── Step C: Load raw pages ─────────────────────────────────────────────
    pages = load_document(filepath)
    if not pages:
        print(f"  ⚠️  EMPTY/UNREADABLE: {filename}")
        return {"status": "empty", "chunks_added": 0}

    # ── Step D: Remove repeated headers/footers (PDF only) ────────────────
    if file_type == "pdf":
        pages = remove_repeated_header_footer(pages)

    # ── Step E: Clean text ─────────────────────────────────────────────────
    for p in pages:
        p["text"] = clean_text(p["text"])

    # Remove pages that are empty after cleaning
    pages = [p for p in pages if p["text"]]
    if not pages:
        print(f"  ⚠️  ALL PAGES EMPTY after cleaning: {filename}")
        return {"status": "empty", "chunks_added": 0}

    # ── Step F: Near-duplicate check ──────────────────────────────────────
    is_near_dup, match_name = is_near_duplicate(pages[0]["text"])
    if is_near_dup:
        print(
            f"  ⚠️  NEAR-DUPLICATE WARNING: '{filename}' is very similar to "
            f"'{match_name}' (cosine ≥ {CONFIG['near_dup_threshold']}). "
            f"Proceeding with ingestion as a new version."
        )

    # ── Step G: Determine version number ──────────────────────────────────
    version = get_file_version(REGISTRY, filename)

    # ── Step H: Chunk & deduplicate ───────────────────────────────────────
    chunks = build_chunks(
        pages=pages,
        filepath=filepath,
        file_hash=file_hash,
        file_type=file_type,
        version=version,
        existing_chunk_hashes=EXISTING_CHUNK_HASHES,
    )

    if not chunks:
        print(f"  ⏭️  No new chunks for '{filename}' (all deduplicated).")
        return {"status": "skipped_exact_dup", "chunks_added": 0}

    # ── Step I: Embed ──────────────────────────────────────────────────────
    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts)

    # ── Step J: Insert into ChromaDB (batch of 500 max) ───────────────────
    batch_size = 500
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        collection.add(
            ids=[c["id"] for c in batch],
            documents=[c["text"] for c in batch],
            embeddings=embeddings[i : i + batch_size],
            metadatas=[c["metadata"] for c in batch],
        )

    # Update in-memory hash set so subsequent files in the same run benefit
    for c in chunks:
        EXISTING_CHUNK_HASHES.add(c["metadata"]["chunk_hash"])

    # ── Step K: Update registry ────────────────────────────────────────────
    REGISTRY[file_hash] = {
        "filename":     filename,
        "filepath":     filepath,
        "file_type":    file_type,
        "version":      version,
        "ingested_at":  datetime.now(timezone.utc).isoformat(),
        "chunk_count":  len(chunks),
        "chunk_ids":    [c["id"] for c in chunks],
    }
    save_registry(REGISTRY, CONFIG["registry_path"])

    print(f"  ✅ INGESTED: {filename}  "
          f"[v{version} | {len(chunks)} chunks | hash: {file_hash[:12]}...]")
    return {"status": "ingested", "chunks_added": len(chunks)}


# ─────────────────────────────────────────────────────────────────────────────
# Run the ingestion loop over all discovered files
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Starting ingestion — {len(all_files)} file(s) found")
print(f"{'='*60}\n")

summary = {"ingested": 0, "skipped_exact_dup": 0, "empty": 0, "error": 0, "warned_near_dup": 0}

for fpath in tqdm(all_files, desc="Ingesting", unit="file"):
    result = ingest_file(fpath)
    summary[result["status"]] = summary.get(result["status"], 0) + 1

print(f"\n{'='*60}")
print("  Ingestion complete — Summary")
print(f"{'='*60}")
for key, val in summary.items():
    print(f"   {key}: {val}")
print(f"   Total docs in collection: {collection.count()}")

---
## Cell 12 — Retrieval Test

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Retrieval Test
#   Demonstrates similarity search against the populated collection.
# ─────────────────────────────────────────────────────────────────────────────

def retrieve(
    query: str,
    top_k: int = CONFIG["top_k"],
    where: Optional[dict] = None,
) -> list[dict]:
    """
    Semantic similarity search.

    Args:
        query:  Natural-language question or keyword string.
        top_k:  Number of results to return.
        where:  Optional ChromaDB metadata filter, e.g. {"file_type": "pdf"}

    Returns:
        List of result dicts with keys: text, metadata, similarity_score.
    """
    query_vec = embed_texts([query])

    kwargs = dict(
        query_embeddings=query_vec,
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    if where:
        kwargs["where"] = where

    raw = collection.query(**kwargs)

    results = []
    for doc, meta, dist in zip(
        raw["documents"][0],
        raw["metadatas"][0],
        raw["distances"][0],
    ):
        results.append({
            "text":             doc,
            "metadata":         meta,
            "similarity_score": round(1.0 - dist, 4),  # convert cosine distance
        })
    return results


def print_results(results: list[dict]) -> None:
    """Pretty-print retrieval results."""
    for i, r in enumerate(results, 1):
        m = r["metadata"]
        print(f"\n{'─'*60}")
        print(f"  Result #{i}   score={r['similarity_score']}")
        print(f"  File   : {m.get('filename')}  (v{m.get('version')})")
        print(f"  Type   : {m.get('file_type')}   Page: {m.get('page')}")
        print(f"  Hash   : {m.get('file_hash','')[:16]}...")
        print(f"  Ingested: {m.get('ingestion_ts')}")
        print(f"  Preview: {r['text'][:300].replace(chr(10), ' ')} ...")
    print(f"\n{'─'*60}")


# ── Run a sample query ────────────────────────────────────────────────────────
# ⬇️ Change this query to something relevant to YOUR documents
SAMPLE_QUERY = "What is the main topic of the documents?"

print(f"Query: \"{SAMPLE_QUERY}\"")
print(f"Top-{CONFIG['top_k']} results:\n")

if collection.count() == 0:
    print("⚠️  Collection is empty. Ingest some files first (Cell 11).")
else:
    results = retrieve(SAMPLE_QUERY)
    print_results(results)

---
## Cell 13 — RAG Function: Retrieval → Prompt Builder

This function is the final building block of a RAG system.
It retrieves relevant chunks and formats a **context-enriched prompt**
ready to be sent to any LLM (OpenAI, Anthropic, Ollama, etc.).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RAG Function
# ─────────────────────────────────────────────────────────────────────────────

RAG_SYSTEM_PROMPT = """\
You are a knowledgeable assistant. Answer the user's question using ONLY the \
context provided below. If the context does not contain enough information, \
say so clearly instead of guessing.
"""


def build_rag_prompt(
    query: str,
    top_k: int = CONFIG["top_k"],
    where: Optional[dict] = None,
    max_context_chars: int = 4000,
) -> dict:
    """
    End-to-end RAG function.

    1. Retrieves top-k relevant chunks from ChromaDB.
    2. Formats retrieved chunks into a numbered context block.
    3. Returns a dict with:
       - system_prompt : str  — instruct the LLM how to behave
       - user_prompt   : str  — context + question ready for the LLM
       - sources       : list — metadata for citations / traceability
       - retrieved_chunks: list — raw results for debugging

    Drop the returned dict straight into any LLM SDK::

        # OpenAI example
        rag = build_rag_prompt("Explain X")
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system",  "content": rag["system_prompt"]},
                {"role": "user",    "content": rag["user_prompt"]},
            ],
        )

        # Anthropic example
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            system=rag["system_prompt"],
            messages=[{"role": "user", "content": rag["user_prompt"]}],
            max_tokens=1024,
        )
    """
    if collection.count() == 0:
        return {
            "system_prompt": RAG_SYSTEM_PROMPT,
            "user_prompt": f"Question: {query}\n\nContext: [No documents ingested yet.]",
            "sources": [],
            "retrieved_chunks": [],
        }

    # ── 1. Retrieve ───────────────────────────────────────────────────────
    chunks = retrieve(query, top_k=top_k, where=where)

    # ── 2. Format context block (respect max_context_chars budget) ─────────
    context_parts = []
    sources = []
    char_budget = max_context_chars

    for idx, chunk in enumerate(chunks, 1):
        m = chunk["metadata"]
        header = (
            f"[{idx}] Source: {m.get('filename')} "
            f"(page {m.get('page')}, v{m.get('version')}, "
            f"score={chunk['similarity_score']})"
        )
        body = chunk["text"]

        entry = f"{header}\n{body}"

        if len(entry) > char_budget:
            # Truncate to remaining budget
            entry = entry[:char_budget] + " [...truncated]"
            context_parts.append(entry)
            sources.append(m)
            break

        context_parts.append(entry)
        sources.append(m)
        char_budget -= len(entry)

    context_str = "\n\n".join(context_parts)

    # ── 3. Assemble prompt ────────────────────────────────────────────────
    user_prompt = (
        f"Context (retrieved from document store):\n"
        f"{'─'*50}\n"
        f"{context_str}\n"
        f"{'─'*50}\n\n"
        f"Question: {query}\n\n"
        f"Please answer based solely on the context above."
    )

    return {
        "system_prompt":    RAG_SYSTEM_PROMPT,
        "user_prompt":      user_prompt,
        "sources":          sources,
        "retrieved_chunks": chunks,
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
rag_output = build_rag_prompt(SAMPLE_QUERY)

print("=" * 60)
print("SYSTEM PROMPT")
print("=" * 60)
print(rag_output["system_prompt"])

print("\n" + "=" * 60)
print("USER PROMPT (first 1200 chars)")
print("=" * 60)
print(rag_output["user_prompt"][:1200])

print("\n" + "=" * 60)
print(f"SOURCES ({len(rag_output['sources'])} chunk(s))")
print("=" * 60)
for s in rag_output["sources"]:
    print(f"  • {s.get('filename')}  page={s.get('page')}  v{s.get('version')}")

print("\n✅ RAG prompt ready. Pass system_prompt + user_prompt to your LLM of choice.")

---
## Cell 14 — Registry Inspection & Pipeline Stats

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Inspect the ingestion registry and collection stats
# Useful for debugging, auditing, or monitoring.
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("  INGESTION REGISTRY")
print("=" * 60)

current_registry = load_registry(CONFIG["registry_path"])

if not current_registry:
    print("  Registry is empty. Run Cell 11 to ingest files.")
else:
    for file_hash, info in current_registry.items():
        print(
            f"  [{file_hash[:12]}...]  "
            f"{info['filename']}  "
            f"v{info['version']}  "
            f"{info['chunk_count']} chunks  "
            f"ingested: {info['ingested_at'][:19]}"
        )

print(f"\n  Total tracked files : {len(current_registry)}")
print(f"  Total chunks in DB  : {collection.count()}")


# ── Show version history for files that were re-ingested ─────────────────────
from collections import defaultdict

version_map: dict = defaultdict(list)
for fhash, info in current_registry.items():
    version_map[info["filename"]].append(info["version"])

versioned = {k: v for k, v in version_map.items() if max(v) > 1}
if versioned:
    print("\n  Files with multiple versions:")
    for fname, versions in versioned.items():
        print(f"    {fname}: versions {sorted(versions)}")
else:
    print("\n  No files with multiple versions found.")

---
## Cell 15 — Utility: Delete a File's Chunks from ChromaDB

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Utility: Remove a specific file from the vector DB and registry.
# Useful when a file is deleted or needs to be fully replaced.
# ─────────────────────────────────────────────────────────────────────────────

def delete_file_from_db(filename: str) -> int:
    """
    Remove all chunks for `filename` from ChromaDB and the registry.

    Returns the number of chunks deleted.
    """
    reg = load_registry(CONFIG["registry_path"])

    # Find all registry entries for this filename
    matching_hashes = [
        fhash for fhash, info in reg.items()
        if info["filename"] == filename
    ]

    if not matching_hashes:
        print(f"  '{filename}' not found in registry.")
        return 0

    total_deleted = 0
    for fhash in matching_hashes:
        chunk_ids = reg[fhash].get("chunk_ids", [])
        if chunk_ids:
            collection.delete(ids=chunk_ids)
            total_deleted += len(chunk_ids)
        del reg[fhash]

    save_registry(reg, CONFIG["registry_path"])

    # Refresh in-memory state
    global REGISTRY, EXISTING_CHUNK_HASHES
    REGISTRY = reg
    EXISTING_CHUNK_HASHES = get_existing_chunk_hashes()

    print(f"  🗑️  Deleted {total_deleted} chunk(s) for '{filename}'.")
    return total_deleted


# ── Example usage (uncomment to run) ─────────────────────────────────────────
# delete_file_from_db("my_old_document.pdf")

print("✅ delete_file_from_db() utility ready.")
print("   Uncomment the example line above to test deletion.")

---
## Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│                    RAG Ingestion Pipeline                    │
│                                                             │
│  ./data/                                                    │
│  ├── *.pdf   ──► load_pdf()  ──► pages[]                   │
│  ├── *.txt   ──► load_txt()  ──►  │                        │
│  └── *.docx  ──► load_docx() ──►  │                        │
│                                   ▼                        │
│                           clean_text()                      │
│                           remove_repeated_header_footer()   │
│                                   │                        │
│            SHA256 file hash ──────┤                        │
│            Registry check ────────┤ ── skip if seen        │
│            Near-dup check ─────────┤ ── warn if similar    │
│                                   │                        │
│                           RecursiveCharacterTextSplitter    │
│                           chunk_size=800 / overlap=100      │
│                                   │                        │
│                     SHA256 chunk hash ── skip if seen      │
│                                   │                        │
│                   SentenceTransformer (all-MiniLM-L6-v2)   │
│                   384-dim L2-normalised embeddings          │
│                                   │                        │
│                          ChromaDB (Docker :8000)            │
│                          collection: rag_collection         │
│                          cosine similarity index            │
│                                   │                        │
│                     retrieve() ─► build_rag_prompt()        │
│                                   │                        │
│                              LLM of choice                  │
└─────────────────────────────────────────────────────────────┘
```

### Data Stock Problem Handling

| Problem | Solution |
|---|---|
| Exact duplicate file | SHA-256 file hash → registry check → skip |
| Near-duplicate file | Embedding cosine similarity query → warn + version |
| Re-ingestion of same dataset | Registry JSON persists across restarts |
| Duplicate chunks across files | SHA-256 chunk hash → skip before insertion |
| File versioning | `get_file_version()` auto-increments per filename |
| Corrupt/empty files | Try/except in loaders → graceful skip |